# 📦 Dataset Trimmer — Emotion Chatbot Project
Trims 3 datasets to under 2,000 rows each, with stratified sampling to keep class balance.

**Datasets:**
- EmpatheticDialogues (Facebook AI)
- Spotify Lyrics Dataset
- GoEmotions (Google)

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# Target row count
TARGET_ROWS = 1800  # safely below 2000
RANDOM_SEED = 42

print('Libraries loaded ✅')
print(f'Target rows per dataset: {TARGET_ROWS}')


---
## 🔍 Step 1 — Explore Available Files

In [ ]:
base_path = '/kaggle/input'

# List all CSV files across all datasets
all_csv = glob.glob(f'{base_path}/**/*.csv', recursive=True)
all_tsv = glob.glob(f'{base_path}/**/*.tsv', recursive=True)

print('=== CSV files found ===')
for f in all_csv:
    print(f)

print('\n=== TSV files found ===')
for f in all_tsv:
    print(f)

---
## 🎵 Dataset 1 — Spotify Lyrics Dataset

In [ ]:
# --- Load Spotify Lyrics ---
spotify_path = '/kaggle/input/datasets/evabot/spotify-lyrics-dataset'

print('Files in Spotify folder:')
for f in glob.glob(f'{spotify_path}/**/*', recursive=True):
    print(f)

In [ ]:
spot_files = glob.glob(f'{spotify_path}/**/*.csv', recursive=True)
print(f'Found: {spot_files}')

df_spot = pd.read_csv(spot_files[0])

print(f'\nOriginal shape: {df_spot.shape}')
print(f'Columns: {list(df_spot.columns)}')
df_spot.head(3)

In [ ]:
# --- Trim Spotify Lyrics ---
# Detect mood/genre column for stratified sampling
mood_col_spot = None
for candidate in ['mood', 'genre', 'tag', 'label', 'category']:
    if candidate in df_spot.columns:
        mood_col_spot = candidate
        break

print(f'Mood/genre column detected: {mood_col_spot}')

if mood_col_spot and df_spot[mood_col_spot].nunique() > 1:
    n_classes = df_spot[mood_col_spot].nunique()
    per_class = TARGET_ROWS // n_classes
    print(f'Classes: {n_classes} | Rows per class: {per_class}')

    df_spot_trimmed = (
        df_spot.groupby(mood_col_spot, group_keys=False)
        .apply(lambda x: x.sample(min(len(x), per_class), random_state=RANDOM_SEED))
        .reset_index(drop=True)
    )
else:
    df_spot_trimmed = df_spot.sample(n=min(TARGET_ROWS, len(df_spot)), random_state=RANDOM_SEED).reset_index(drop=True)

# Drop rows with missing lyrics if column exists
for col in ['lyrics', 'lyric', 'text']:
    if col in df_spot_trimmed.columns:
        before = len(df_spot_trimmed)
        df_spot_trimmed = df_spot_trimmed.dropna(subset=[col]).reset_index(drop=True)
        print(f'Dropped {before - len(df_spot_trimmed)} rows with missing {col}')
        break

print(f'\nOriginal rows : {len(df_spot)}')
print(f'Trimmed rows  : {len(df_spot_trimmed)}')
df_spot_trimmed.head(3)

---
## 😊 Dataset 2 — GoEmotions (Google)

**Source:** `google-research-datasets/go_emotions` (official, Apache 2.0 license)  
**Config used:** `simplified` — has `text`, `labels` (list of emotion IDs), and `id`  

Emotion ID mapping (0–27):  
`admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire,
disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief,
joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, surprise, neutral`

In [ ]:
# Install datasets library if not already available
!pip install datasets -q

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset

RANDOM_SEED  = 42
TARGET_ROWS  = 1800  # safely under 2000

# ── Kaggle output directory (guaranteed to exist) ──────────────────────────
OUTPUT_DIR  = '/kaggle/working'
OUTPUT_FILE = 'goemotions_trimmed.csv'
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
os.makedirs(OUTPUT_DIR, exist_ok=True)   # no-op on Kaggle, safety net locally

# Full emotion label map (ID → name)
EMOTION_LABELS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval',
    'caring', 'confusion', 'curiosity', 'desire', 'disappointment',
    'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear',
    'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism',
    'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

print('✅ Libraries loaded')
print(f'📁 Output will be saved to: {OUTPUT_PATH}')

---
## 🔽 Step 1 — Load Dataset from HuggingFace

In [ ]:
print('Loading GoEmotions from HuggingFace...')

dataset = load_dataset('google-research-datasets/go_emotions', 'simplified')

print('\nSplits available:', list(dataset.keys()))
print('Train size :', len(dataset['train']))
print('Val size   :', len(dataset['validation']))
print('Test size  :', len(dataset['test']))

In [ ]:
# Merge all splits into one dataframe
df_train = dataset['train'].to_pandas()
df_val   = dataset['validation'].to_pandas()
df_test  = dataset['test'].to_pandas()

df = pd.concat([df_train, df_val, df_test], ignore_index=True)

print(f'Total rows after merge: {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head(3)

---
## 🔎 Step 2 — Inspect & Understand the Data

In [ ]:
# The 'labels' column is a list of emotion IDs per row
# Example: [0] = admiration only | [2, 6] = anger + confusion

print('Sample labels column values:')
print(df['labels'].head(10).tolist())

# Extract primary label (first ID in the list)
df['primary_label_id']   = df['labels'].apply(lambda x: x[0] if len(x) > 0 else 27)  # default: neutral
df['primary_label_name'] = df['primary_label_id'].apply(lambda x: EMOTION_LABELS[x])

print('\nEmotion distribution (top 10):')
print(df['primary_label_name'].value_counts().head(10))

---
## ✂️ Step 3 — Stratified Trim to 1,800 Rows

In [ ]:
n_classes = df['primary_label_name'].nunique()
per_class = TARGET_ROWS // n_classes

print(f'Unique emotion classes : {n_classes}')
print(f'Rows sampled per class : {per_class}')
print(f'Expected total rows    : ~{per_class * n_classes}')

df_trimmed = (
    df.groupby('primary_label_name', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), per_class), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

print(f'\n✅ Trimmed shape: {df_trimmed.shape}')
print(f'Under 2000 rows: {len(df_trimmed) < 2000}')

In [ ]:
# Preview emotion balance after trimming
print('Emotion distribution after trimming:')
print(df_trimmed['primary_label_name'].value_counts())

---
## 🧹 Step 4 — Clean Up Columns for Export

In [ ]:
# Keep only the columns useful for your NLP project
df_export = df_trimmed[['id', 'text', 'labels', 'primary_label_id', 'primary_label_name']].copy()

# Convert labels list to string for CSV compatibility
df_export['labels'] = df_export['labels'].apply(lambda x: str(x))

# Drop rows with missing text (safety check)
before = len(df_export)
df_export = df_export.dropna(subset=['text']).reset_index(drop=True)
print(f'Dropped {before - len(df_export)} rows with missing text')

print(f'\nFinal export shape: {df_export.shape}')
df_export.head(5)

---
## 💾 Step 5 — Save & Verify CSV in Kaggle Output

In [ ]:
# ── Save to /kaggle/working so Kaggle picks it up as an Output file ────────
df_export.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

# ── Verify the file actually landed on disk ────────────────────────────────
assert os.path.isfile(OUTPUT_PATH), f'❌ File not found at {OUTPUT_PATH}'

size_kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f'✅ Saved : {OUTPUT_PATH}')
print(f'   Rows  : {len(df_export):,}')
print(f'   Size  : {size_kb:.1f} KB')
print()

# ── List /kaggle/working so you can confirm it appears ────────────────────
print('📂 Files currently in /kaggle/working:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    fsize = os.path.getsize(fpath) / 1024
    print(f'   {f:<40}  {fsize:>8.1f} KB')

print()
print('👉 Open the  Output  tab (right panel) → you should see goemotions_trimmed.csv')

In [ ]:
# Final validation summary
print('='*45)
print('FINAL SUMMARY')
print('='*45)
status = '✅ PASS' if len(df_export) < 2000 else '❌ OVER LIMIT'
print(f'{status} — {len(df_export):,} rows × {len(df_export.columns)} columns')
print(f'Columns : {list(df_export.columns)}')
print(f'Saved to: {OUTPUT_PATH}')
print('='*45)

# Quick preview
df_export.sample(5, random_state=42)